In [12]:
import json
import lzma
from pathlib import Path
import pandas as pd


def load_json_xz(path):
    with lzma.open(path, "rt", encoding="utf-8") as fh:
        return pd.DataFrame(json.load(fh))


files = sorted(Path("data").glob("*.json.xz"))
df = pd.concat((load_json_xz(path) for path in files), ignore_index=True)
df.shape

(70974, 169)

In [14]:
def extract_ticker_prices(row):
    tickers = row.get('mentioned_companies')
    if not isinstance(tickers, list) or len(tickers) == 0:
        return []

    ticker_rows = []
    for ticker in tickers:
        prev_price = row.get(f'prev_day_price_{ticker}')
        curr_price = row.get(f'curr_day_price_{ticker}')
        next_price = row.get(f'next_day_price_{ticker}')

        # Keep a ticker only when we can compute its movement target.
        if pd.notna(prev_price) and pd.notna(next_price):
            ticker_rows.append({
                'ticker': ticker,
                'prev_price': prev_price,
                'curr_price': curr_price,
                'next_price': next_price,
            })

    return ticker_rows

In [22]:
keep_cols = ["title", "maintext", "date_publish", "mentioned_companies", "sentiment", "emotion", "named_entities"]

# Keep the original article index after exploding so rows remain aligned.
ticker_data = df.apply(extract_ticker_prices, axis=1).explode().dropna()

len(ticker_data)

85957

In [21]:
ticker_data

[{'ticker': 'AAPL',
  'prev_price': 143.64999,
  'curr_price': 146.58,
  'next_price': 147.50999},
 {'ticker': 'MA',
  'prev_price': 116.61,
  'curr_price': 116.41,
  'next_price': 116.65},
 {'ticker': 'T',
  'prev_price': 41.06,
  'curr_price': 41.12,
  'next_price': 41.21},
 {'ticker': 'VZ',
  'prev_price': 48.03,
  'curr_price': 48.04,
  'next_price': 48.37},
 {'ticker': 'GOOGL',
  'prev_price': 940.48999,
  'curr_price': 917.78998,
  'next_price': 908.72998},
 {'ticker': 'C',
  'prev_price': 72.28,
  'curr_price': 72.65,
  'next_price': 72.74},
 {'ticker': 'BABA',
  'prev_price': 178.89,
  'curr_price': 179.2,
  'next_price': 182.09},
 {'ticker': 'AAPL',
  'prev_price': 121.63,
  'curr_price': 121.35,
  'next_price': 128.75},
 {'ticker': 'GOOGL',
  'prev_price': 802.32001,
  'curr_price': 796.78998,
  'next_price': 795.69501},
 {'ticker': 'AMZN',
  'prev_price': 830.38,
  'curr_price': 823.47998,
  'next_price': 832.34998},
 {'ticker': 'NFLX',
  'prev_price': 141.22,
  'curr_price'

In [23]:
if ticker_data.empty:
    df_clean = pd.DataFrame(columns=keep_cols + ["ticker", "prev_price", "curr_price", "next_price"])
else:
    ticker_df = pd.DataFrame(ticker_data.tolist(), index=ticker_data.index)
    df_clean = (
        df.loc[ticker_df.index, keep_cols]
        .reset_index(drop=True)
        .join(ticker_df.reset_index(drop=True))
    )

df_clean.shape

(85957, 11)

In [24]:
df_clean.head()

,title,maintext,date_publish,mentioned_companies,sentiment,emotion,named_entities,ticker,prev_price,curr_price,next_price
0,LEAKED PHOTOS: Fitbit’s new headphones and tro...,Yahoo Finance has obtained photos of Fitbit’s ...,2017-05-01 14:04:36,[AAPL],"{'negative': 0.004725542385131121, 'neutral': ...","{'neutral': 0.9596450924873352, 'surprise': 0....","[{'entity_group': 'MISC', 'word': 'San Francis...",AAPL,143.64999,146.58000,147.50999
1,Testosterone is the enemy of smart investing d...,The pitfalls of the ‘hurry up’ hormone\nShutte...,2017-05-09 08:50:32,[MA],"{'negative': 0.002294974634423852, 'neutral': ...","{'neutral': 0.7487770915031433, 'surprise': 0....","[{'entity_group': 'MISC', 'word': 'Shut', 'nor...",MA,116.61000,116.41000,116.65000
2,Open-internet rules look dead. Now what?,President Trump’s new head of the Federal Comm...,2017-02-07 23:16:13,"[T, VZ]","{'negative': 0.031167231500148773, 'neutral': ...","{'neutral': 0.8945390582084656, 'surprise': 0....","[{'entity_group': 'PER', 'word': 'Trump', 'nor...",T,41.06000,41.12000,41.21000
3,Open-internet rules look dead. Now what?,President Trump’s new head of the Federal Comm...,2017-02-07 23:16:13,"[T, VZ]","{'negative': 0.031167231500148773, 'neutral': ...","{'neutral': 0.8945390582084656, 'surprise': 0....","[{'entity_group': 'PER', 'word': 'Trump', 'nor...",VZ,48.03000,48.04000,48.37000
4,A ruling against Google in Canada could affect...,The Supreme Court of Canada issued an order to...,2017-06-29 20:45:02,[GOOGL],"{'negative': 0.03922323137521744, 'neutral': 0...","{'neutral': 0.3601921796798706, 'anger': 0.305...","[{'entity_group': 'ORG', 'word': 'Google', 'no...",GOOGL,940.48999,917.78998,908.72998
